In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

processed_path = Path("../data/processed")

amazon_pairs = pd.read_parquet(
    processed_path / "amazon_support_pairs.parquet"
)

print("Shape:", amazon_pairs.shape)
amazon_pairs.head()

Shape: (166963, 18)


,support_tweet_id,support_author,inbound_support,support_created_at,support_text,response_tweet_id,in_response_to_tweet_id,text_length,customer_tweet_id,tweet_id_customer,customer_author,customer_inbound,customer_text,customer_created_at,customer_text_clean,clean_length,mention_only,low_information
0,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272,126,272,272,115770,True,amazonのfireTVstickが見れない😢,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢,24,False,False
1,273,AmazonHelp,False,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271,69,271,271,115770,True,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,Wed Nov 22 09:30:36 +0000 2017,電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんで...,51,False,False
2,275,AmazonHelp,False,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274,54,274,274,115770,True,@AmazonHelp こちらこそありがとうございました。,Wed Nov 22 09:44:04 +0000 2017,こちらこそありがとうございました。,17,False,False
3,324,AmazonHelp,False,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325,153,325,325,115792,True,amazonプライムビデオ、再生エラーが多いです,Wed Nov 22 08:55:35 +0000 2017,amazonプライムビデオ、再生エラーが多いです,24,False,False
4,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617,133,617,617,115820,True,Way to drop the ball on customer service @1158...,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service so pi...,61,False,False


In [3]:
intent_prototypes = {
    "DELIVERY_DELAY":
        "my package delivery is late or delayed",

    "DELIVERY_NOT_RECEIVED":
        "my package was not received or is missing",

    "DELIVERY_TRACKING":
        "where is my package tracking status",

    "DELIVERY_DRIVER_ISSUE":
        "problem with delivery driver or delivery attempt",

    "ORDER_STATUS":
        "what is the status of my order",

    "ORDER_CANCELLATION":
        "I want to cancel my order",

    "MISSING_ITEM":
        "an item is missing from my delivered order",

    "WRONG_ITEM_RECEIVED":
        "I received the wrong item",

    "RETURN_REPLACEMENT":
        "I want to return or replace a product",

    "REFUND_STATUS":
        "where is my refund",

    "ACCOUNT_ACCESS":
        "I cannot access or log into my account",

    "ACCOUNT_SECURITY":
        "my account has been hacked or compromised",

    "UNEXPECTED_CHARGE":
        "I was charged unexpectedly or charged twice",

    "CASHBACK":
        "I have not received my cashback",

    "PRODUCT_ISSUE":
        "my product is broken defective damaged or not working"
}

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_multi = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 855.59it/s]


In [5]:
prototype_embeddings = model_multi.encode(
    list(intent_prototypes.values()),
    normalize_embeddings=True
)

In [6]:
candidate_sample = amazon_pairs.sample(
    n=10000,
    random_state=42
).copy()

candidate_embeddings = model_multi.encode(
    candidate_sample["customer_text_clean"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 313/313 [02:24<00:00,  2.16it/s]


In [7]:
intent_candidates = {}

for i, intent in enumerate(intent_prototypes):

    similarities = cosine_similarity(
        prototype_embeddings[i].reshape(1, -1),
        candidate_embeddings
    )[0]

    top_indices = similarities.argsort()[-40:][::-1]

    result = candidate_sample.iloc[top_indices].copy()
    result["similarity"] = similarities[top_indices]

    intent_candidates[intent] = result

In [8]:
intent_candidates["DELIVERY_DELAY"][
    ["customer_text_clean", "similarity"]
].head(40)

,customer_text_clean,similarity
25731,"hello, My package was out for delivery but now...",0.824175
34460,"Thankyou. Turns out delivery is late, should b...",0.776511
97154,Delivery date pending,0.774415
91358,Hi My package has missed two delivery dates bu...,0.755127
102504,Delivery date is already delayed. 111-2691482-...,0.747700
18142,can you update me on my delivery that was alre...,0.747494
90432,my package that was to arrive today still says...,0.740526
97237,Another expected delivery today and another de...,0.739923
31049,ich warte seit dem 4.10 auf eine Lieferung. Am...,0.724916
68726,No and no;the estimated delivery date is usele...,0.721474


In [9]:
intent_candidates["MISSING_ITEM"][
    ["customer_text_clean", "similarity"]
].head(40)

,customer_text_clean,similarity
32466,I am not able to return my Order. Even though ...,0.740536
70811,"Order # 402-5479722-9566747, wrong item receiv...",0.735128
134422,My Order id: 403-6019015-9546740.I still haven...,0.727035
149835,Ordered something that never showed. Emailed t...,0.726005
92563,Thanks for messing up my order just because I ...,0.725914
35,my order hasn’t arrived (it was due 14th Octob...,0.724240
161722,hi my order is showing as delivered but it isn...,0.721112
3870,The item is being returned undelivered by the ...,0.712112
57863,Order:408-4742078-0005944 I HAVE NOT RECEIVED ...,0.711598
47721,"Lies, I ordered this product and did not recei...",0.709786


In [10]:
pd.set_option("display.max_colwidth", None)

delivery_delay_candidates = intent_candidates["DELIVERY_DELAY"][
    ["customer_text_clean", "similarity"]
].head(40)

display(delivery_delay_candidates)

,customer_text_clean,similarity
25731,"hello, My package was out for delivery but now it is saying it is delayed because courier was unable to arrange delivery",0.824175
34460,"Thankyou. Turns out delivery is late, should be due 1-4 Dec. Disappointing",0.776511
97154,Delivery date pending,0.774415
91358,Hi My package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday?,0.755127
102504,"Delivery date is already delayed. 111-2691482-1933821 , also it says that it is a customer initiated delay which isn't right.",0.747700
18142,can you update me on my delivery that was already delayed and now even later than you said?,0.747494
90432,my package that was to arrive today still says out for delivery-it’s 930pm. Is it coming tonight? Need it for tomorrow.,0.740526
97237,Another expected delivery today and another delay courtesy of AMZL US!,0.739923
31049,"ich warte seit dem 4.10 auf eine Lieferung. Am 8.10 wurde sie durch einen Mitarbeiter verschoben, wann bekomme ich mein Paket?",0.724916
68726,No and no;the estimated delivery date is useless because only going to be in the us a few hours. The package was “guaranteed” for yesterday,0.721474


In [11]:
for idx, row in delivery_delay_candidates.iterrows():
    print(f"\n[{idx}] similarity={row['similarity']:.3f}")
    print(row["customer_text_clean"])


[25731] similarity=0.824
hello, My package was out for delivery but now it is saying it is delayed because courier was unable to arrange delivery

[34460] similarity=0.777
Thankyou. Turns out delivery is late, should be due 1-4 Dec. Disappointing

[97154] similarity=0.774
Delivery date pending

[91358] similarity=0.755
Hi My package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday?

[102504] similarity=0.748
Delivery date is already delayed. 111-2691482-1933821 , also it says that it is a customer initiated delay which isn't right.

[18142] similarity=0.747
can you update me on my delivery that was already delayed and now even later than you said?

[90432] similarity=0.741
my package that was to arrive today still says out for delivery-it’s 930pm. Is it coming tonight? Need it for tomorrow.

[97237] similarity=0.740
Another expected delivery today and another delay courtesy of AMZL US!

[31049] similarity=0.725
ich wart

In [12]:
golden_candidates = []

for intent, candidates in intent_candidates.items():
    temp = candidates.copy()

    temp["candidate_intent"] = intent
    temp["human_intent"] = ""
    temp["difficulty"] = ""
    temp["notes"] = ""

    golden_candidates.append(temp)

golden_candidates = pd.concat(
    golden_candidates,
    ignore_index=True
)

golden_candidates = golden_candidates[
    [
        "customer_tweet_id",
        "customer_text",
        "customer_text_clean",
        "candidate_intent",
        "similarity",
        "human_intent",
        "difficulty",
        "notes"
    ]
]

golden_candidates.head()

,customer_tweet_id,customer_text,customer_text_clean,candidate_intent,similarity,human_intent,difficulty,notes
0,351833,"@AmazonHelp hello, My package was out for delivery but now it is saying it is delayed because courier was unable to arrange delivery","hello, My package was out for delivery but now it is saying it is delayed because courier was unable to arrange delivery",DELIVERY_DELAY,0.824175,,,
1,448256,"@AmazonHelp Thankyou. Turns out delivery is late, should be due 1-4 Dec. Disappointing","Thankyou. Turns out delivery is late, should be due 1-4 Dec. Disappointing",DELIVERY_DELAY,0.776511,,,
2,1484058,Delivery date pending https://t.co/eFKBoj7LA7,Delivery date pending,DELIVERY_DELAY,0.774415,,,
3,1385298,Hi @115830\nMy package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday? https://t.co/YGr1AdoFt3,Hi My package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday?,DELIVERY_DELAY,0.755127,,,
4,1575619,"@AmazonHelp Delivery date is already delayed. 111-2691482-1933821 , also it says that it is a customer initiated delay which isn't right.","Delivery date is already delayed. 111-2691482-1933821 , also it says that it is a customer initiated delay which isn't right.",DELIVERY_DELAY,0.747700,,,


### Start building candidate pool

In [13]:
# Take the top 25 semantic candidates for every intent
candidate_parts = []

for intent, candidates in intent_candidates.items():
    temp = candidates.head(25).copy()
    temp["candidate_intent"] = intent
    candidate_parts.append(temp)

semantic_pool = pd.concat(candidate_parts, ignore_index=True)

print("Before deduplication:", len(semantic_pool))
print("Unique tweets:", semantic_pool["customer_tweet_id"].nunique())

Before deduplication: 375
Unique tweets: 338


In [14]:
semantic_pool = (
    semantic_pool
    .sort_values("similarity", ascending=False)
    .drop_duplicates("customer_tweet_id")
    .reset_index(drop=True)
)

print("After deduplication:", len(semantic_pool))

After deduplication: 338


In [15]:
random_pool = amazon_pairs.sample(
    n=100,
    random_state=123
).copy()

random_pool["candidate_intent"] = "RANDOM"
random_pool["similarity"] = np.nan

In [16]:
random_pool = random_pool[
    [
        "customer_tweet_id",
        "customer_text",
        "customer_text_clean",
        "candidate_intent",
        "similarity"
    ]
]

In [17]:
semantic_pool = semantic_pool[
    [
        "customer_tweet_id",
        "customer_text",
        "customer_text_clean",
        "candidate_intent",
        "similarity"
    ]
]

candidate_pool = pd.concat(
    [semantic_pool, random_pool],
    ignore_index=True
)

candidate_pool = (
    candidate_pool
    .drop_duplicates("customer_tweet_id")
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Final candidate pool:", len(candidate_pool))

Final candidate pool: 438


In [18]:
candidate_pool["human_intent"] = ""
candidate_pool["difficulty"] = ""
candidate_pool["notes"] = ""

In [19]:
display(
    candidate_pool[
        [
            "customer_tweet_id",
            "customer_text",
            "candidate_intent",
            "similarity",
            "human_intent",
            "difficulty",
            "notes"
        ]
    ].head(30)
)

,customer_tweet_id,customer_text,candidate_intent,similarity,human_intent,difficulty,notes
0,2797118,@AmazonHelp Hola! Son vendidos directamente por Amazon.,RANDOM,NaN,,,
1,1222599,@115821 seriously disappointed in your lack of security. My account was hacked and nobody is in a hurry to help me :(,ACCOUNT_SECURITY,0.739902,,,
2,376282,@AmazonHelp I was ordered 2 no’s above item but I received only one short buy yesterday https://t.co/AmZ1uxwd0I,WRONG_ITEM_RECEIVED,0.576438,,,
3,1087661,@AmazonHelp https://t.co/8aZ3NUAScP. 🤘🤘,RANDOM,NaN,,,
4,2972777,今更ながらアマゾンビデオで動画ダウンロードしておけば外でも通信量なしで観れるの気づいたので、昼休みとか利用してアニメ観よう。まずはメイドインアビス,RANDOM,NaN,,,
5,233785,@AmazonHelp Tracking on Canada Post says delivered on the 29th. Not updated on Amazon though.,DELIVERY_TRACKING,0.552647,,,
6,314242,@115850 Pathetic service! Its been 10 days to receive my order still cash back isn't received. No call connect in Customer Care,CASHBACK,0.655791,,,
7,2856569,"@AmazonHelp hola! Hice un pedido de dos artículos, uno en preventa y otro fuera de stock, he cancelado el del stock y creo que el de preventa también pero ese no quería cancelarlo. Me dice que se está preparando la cancelación pero quiero volvero a pedir, cuando podré?",ORDER_CANCELLATION,0.736567,,,
8,236254,@AmazonHelp V r talking abt Mumbai- r u saying dt largest e-commerce player can't deliver products in a metro city like Mumbai? Contd..,RANDOM,NaN,,,
9,250577,"shiiiiit I love how amazon is like, “the delivery driver is 5 minutes away from your house” \n\nthey’re most consistent than the girls I’ve talked to",RANDOM,NaN,,,


In [20]:
golden_candidate_path = Path("../data/golden")

golden_candidate_path.mkdir(
    parents=True,
    exist_ok=True
)

candidate_pool.to_csv(
    golden_candidate_path / "golden_candidate_pool.csv",
    index=False
)

print("Saved:", golden_candidate_path / "golden_candidate_pool.csv")

Saved: ..\data\golden\golden_candidate_pool.csv


In [21]:
candidate_pool["candidate_intent"].value_counts()

candidate_intent
RANDOM                   100
ORDER_CANCELLATION        25
MISSING_ITEM              25
ORDER_STATUS              25
REFUND_STATUS             25
UNEXPECTED_CHARGE         25
DELIVERY_NOT_RECEIVED     25
DELIVERY_DELAY            24
CASHBACK                  23
ACCOUNT_ACCESS            22
DELIVERY_DRIVER_ISSUE     22
DELIVERY_TRACKING         21
PRODUCT_ISSUE             21
ACCOUNT_SECURITY          19
WRONG_ITEM_RECEIVED       18
RETURN_REPLACEMENT        18
Name: count, dtype: int64

In [22]:
candidate_pool["candidate_intent"].value_counts(normalize=True).round(3)

candidate_intent
RANDOM                   0.228
ORDER_CANCELLATION       0.057
MISSING_ITEM             0.057
ORDER_STATUS             0.057
REFUND_STATUS            0.057
UNEXPECTED_CHARGE        0.057
DELIVERY_NOT_RECEIVED    0.057
DELIVERY_DELAY           0.055
CASHBACK                 0.053
ACCOUNT_ACCESS           0.050
DELIVERY_DRIVER_ISSUE    0.050
DELIVERY_TRACKING        0.048
PRODUCT_ISSUE            0.048
ACCOUNT_SECURITY         0.043
WRONG_ITEM_RECEIVED      0.041
RETURN_REPLACEMENT       0.041
Name: proportion, dtype: float64

In [23]:
annotation_df = candidate_pool[
    [
        "customer_tweet_id",
        "customer_text",
        "candidate_intent",
        "similarity"
    ]
].copy()

annotation_df["human_intent"] = ""
annotation_df["difficulty"] = ""
annotation_df["notes"] = ""

annotation_df = annotation_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

annotation_path = Path("../data/golden/amazon_golden_annotation.csv")

annotation_df.to_csv(
    annotation_path,
    index=False
)

print("Saved:", annotation_path)
print("Rows:", len(annotation_df))

Saved: ..\data\golden\amazon_golden_annotation.csv
Rows: 438


In [46]:
batch_1 = annotation_df.iloc[150:199].copy()

display(
    batch_1[
        [
            "customer_tweet_id",
            "customer_text",
            "candidate_intent",
            "similarity",
            "human_intent",
            "difficulty",
            "notes"
        ]
    ]
)

,customer_tweet_id,customer_text,candidate_intent,similarity,human_intent,difficulty,notes
150,928315,I have spend 31k on a product and i get a defective one? @115850 @AmazonHelp &amp; no replacement for it? https://t.co/Rr1d7IHS8P,PRODUCT_ISSUE,0.687471,DELIVERY_NOT_RECEIVED,EASY,Product/package not received despite delivered status
151,1385298,Hi @115830\nMy package has missed two delivery dates but is now refundable if late by a further 10 months? Also Aug 10th isn't a Saturday? https://t.co/YGr1AdoFt3,DELIVERY_DELAY,0.755127,DELIVERY_TRACKING,EASY,Carrier tracking and Amazon status disagree
152,1048775,Another failed delivery from @117795 Getting fed up with this as have lost count of the number of deliveries that have failed,DELIVERY_DRIVER_ISSUE,0.611137,ACCOUNT_ACCESS,EASY,Cannot sign in; password rejected
153,1401461,@115821 so a package was supposed to be delivered today. Tracking last shows it left on Thursday but nothing since and package not at home any ideas?,DELIVERY_TRACKING,0.610237,ACCOUNT_SECURITY,EASY,Account compromise/fraud accusation
154,1283811,@AmazonHelp product delivered not up to the quality standard. Not expected this quality. Want to return! Please help,PRODUCT_ISSUE,0.523920,MISSING_ITEM,BOUNDARY,One item never arrived; refund is secondary
155,685683,"@115830 Now I can't get into my account at all, you're a bunch of thieving Ba****ds!",ACCOUNT_ACCESS,0.661874,REFUND_STATUS,EASY,Asking about refund initiation
156,1205000,@AmazonHelp Just get my package it was suppose to be here today but oh no,DELIVERY_NOT_RECEIVED,0.689905,OTHER,MESSY,Complaint without a clearly supported operational intent
157,1391145,"@AmazonHelp You have charged me twice to two items, my bank confirmed it, I gave the codes to you but still you're denying it. Unbelievable.",UNEXPECTED_CHARGE,0.576666,OTHER,MESSY,Synchronization/account feature issue outside taxonomy
158,2042074,@AmazonHelp Yes but the bank says you have double charged but you refuse to acknowledge that or talk to them. This has been happening frequently.,UNEXPECTED_CHARGE,0.546747,ORDER_CANCELLATION,EASY,Asking why order was cancelled
159,1595329,@AmazonHelp last month my card was charged for prume when even though its canceled and it did it again this month,UNEXPECTED_CHARGE,0.499057,DELIVERY_DELAY,EASY,Explicitly says delivery date is delayed


### Out-of-taxonomy rule

If a message does not clearly correspond to any supported operational intent,
label it `OTHER` rather than forcing it into the closest intent.

`OTHER` is an evaluation/rejection class, not one of the 15 business intents.

Examples:
- praise or compliments
- generic comments without a support request
- unsupported Prime/membership questions
- messages too ambiguous to map reliably

### Primary-intent rule

When a message contains multiple issues, label the intent corresponding to
the customer's primary operational problem.

Examples:

"Tracking says delivered but I never received the package."
→ DELIVERY_NOT_RECEIVED

"My package was supposed to arrive yesterday but tracking hasn't updated."
→ DELIVERY_DELAY

"How do I track the package that I returned?"
→ RETURN_REPLACEMENT

In [47]:
print("Total candidates:", len(annotation_df))
print("Annotated:", annotation_df["human_intent"].ne("").sum())
print("Unannotated:", annotation_df["human_intent"].eq("").sum())

Total candidates: 438
Annotated: 197
Unannotated: 241


In [44]:
for i, (intent, difficulty, note) in enumerate(labels_100_148,start=150):
    annotation_df.loc[i, "human_intent"] = intent
    annotation_df.loc[i, "difficulty"] = difficulty
    annotation_df.loc[i, "notes"] = note

In [48]:
golden_df = annotation_df[
    annotation_df["human_intent"].ne("")
].copy()

print(
    golden_df["human_intent"]
    .value_counts()
)

human_intent
OTHER                    37
ORDER_CANCELLATION       19
REFUND_STATUS            17
DELIVERY_NOT_RECEIVED    16
DELIVERY_TRACKING        16
ACCOUNT_ACCESS           15
PRODUCT_ISSUE            13
ACCOUNT_SECURITY         12
ORDER_STATUS             12
DELIVERY_DELAY           11
RETURN_REPLACEMENT        9
UNEXPECTED_CHARGE         9
CASHBACK                  5
MISSING_ITEM              3
DELIVERY_DRIVER_ISSUE     3
Name: count, dtype: int64


In [49]:
print(
    golden_df["difficulty"]
    .value_counts()
)

difficulty
EASY        123
MESSY        40
BOUNDARY     34
Name: count, dtype: int64


In [50]:
pd.crosstab(
    golden_df["human_intent"],
    golden_df["difficulty"]
)

difficulty,BOUNDARY,EASY,MESSY
human_intent,,,
ACCOUNT_ACCESS,2,13,0
ACCOUNT_SECURITY,0,11,1
CASHBACK,0,5,0
DELIVERY_DELAY,3,6,2
DELIVERY_DRIVER_ISSUE,0,3,0
DELIVERY_NOT_RECEIVED,4,12,0
DELIVERY_TRACKING,3,13,0
MISSING_ITEM,2,1,0
ORDER_CANCELLATION,2,17,0


In [53]:
print(
    "Duplicate tweet IDs:",
    golden_df["customer_tweet_id"].duplicated().sum()
)

print(
    "Duplicate texts:",
    golden_df["customer_text"].duplicated().sum()
)

Duplicate tweet IDs: 0
Duplicate texts: 0


In [55]:
golden_columns = [
    "customer_tweet_id",
    "customer_text",
    "customer_text",
    "human_intent",
    "difficulty",
    "notes"
]

golden_df = golden_df[golden_columns].copy()

golden_path = Path("../data/golden/amazon_golden_set.csv")

golden_df.to_csv(
    golden_path,
    index=False
)

print(f"Saved {len(golden_df)} examples to:")
print(golden_path)

Saved 197 examples to:
..\data\golden\amazon_golden_set.csv


In [56]:
golden_df["human_intent"].value_counts()

human_intent
OTHER                    37
ORDER_CANCELLATION       19
REFUND_STATUS            17
DELIVERY_NOT_RECEIVED    16
DELIVERY_TRACKING        16
ACCOUNT_ACCESS           15
PRODUCT_ISSUE            13
ACCOUNT_SECURITY         12
ORDER_STATUS             12
DELIVERY_DELAY           11
RETURN_REPLACEMENT        9
UNEXPECTED_CHARGE         9
CASHBACK                  5
MISSING_ITEM              3
DELIVERY_DRIVER_ISSUE     3
Name: count, dtype: int64

In [57]:
pd.crosstab(
    golden_df["human_intent"],
    golden_df["difficulty"]
)

difficulty,BOUNDARY,EASY,MESSY
human_intent,,,
ACCOUNT_ACCESS,2,13,0
ACCOUNT_SECURITY,0,11,1
CASHBACK,0,5,0
DELIVERY_DELAY,3,6,2
DELIVERY_DRIVER_ISSUE,0,3,0
DELIVERY_NOT_RECEIVED,4,12,0
DELIVERY_TRACKING,3,13,0
MISSING_ITEM,2,1,0
ORDER_CANCELLATION,2,17,0


In [58]:
golden_df["human_intent"].value_counts().sort_index()

human_intent
ACCOUNT_ACCESS           15
ACCOUNT_SECURITY         12
CASHBACK                  5
DELIVERY_DELAY           11
DELIVERY_DRIVER_ISSUE     3
DELIVERY_NOT_RECEIVED    16
DELIVERY_TRACKING        16
MISSING_ITEM              3
ORDER_CANCELLATION       19
ORDER_STATUS             12
OTHER                    37
PRODUCT_ISSUE            13
REFUND_STATUS            17
RETURN_REPLACEMENT        9
UNEXPECTED_CHARGE         9
Name: count, dtype: int64

In [59]:
remaining = annotation_df[
    annotation_df["human_intent"].eq("")
].copy()

print("Remaining:", len(remaining))

Remaining: 241


In [60]:
rare_candidates = remaining[
    remaining["candidate_intent"].isin([
        "MISSING_ITEM",
        "DELIVERY_DRIVER_ISSUE",
        "CASHBACK",
        "WRONG_ITEM_RECEIVED",
        "RETURN_REPLACEMENT"
    ])
].copy()

print(
    rare_candidates["candidate_intent"]
    .value_counts()
)

candidate_intent
DELIVERY_DRIVER_ISSUE    19
WRONG_ITEM_RECEIVED      14
RETURN_REPLACEMENT       13
MISSING_ITEM             11
CASHBACK                 11
Name: count, dtype: int64


In [63]:
# Add selected rare-intent examples
labels_to_add = {
    201: "MISSING_ITEM",
    381: "MISSING_ITEM",
    426: "MISSING_ITEM",

    222: "DELIVERY_DRIVER_ISSUE",
    244: "DELIVERY_DRIVER_ISSUE",
    262: "DELIVERY_DRIVER_ISSUE",
    312: "DELIVERY_DRIVER_ISSUE",
    361: "DELIVERY_DRIVER_ISSUE",

    205: "CASHBACK",
    255: "CASHBACK",
    334: "CASHBACK",

    254: "WRONG_ITEM_RECEIVED",
    340: "WRONG_ITEM_RECEIVED",
    372: "WRONG_ITEM_RECEIVED",
    377: "WRONG_ITEM_RECEIVED",
    431: "WRONG_ITEM_RECEIVED",
}

for idx, label in labels_to_add.items():
    annotation_df.loc[idx, "human_intent"] = label

In [64]:
golden_df = annotation_df[
    annotation_df["human_intent"].notna() &
    (annotation_df["human_intent"] != "")
].copy()

print("Golden set size:", len(golden_df))
print()
print(golden_df["human_intent"].value_counts())

Golden set size: 213

human_intent
OTHER                    37
ORDER_CANCELLATION       19
REFUND_STATUS            17
DELIVERY_NOT_RECEIVED    16
DELIVERY_TRACKING        16
ACCOUNT_ACCESS           15
PRODUCT_ISSUE            13
ACCOUNT_SECURITY         12
ORDER_STATUS             12
DELIVERY_DELAY           11
RETURN_REPLACEMENT        9
UNEXPECTED_CHARGE         9
CASHBACK                  8
DELIVERY_DRIVER_ISSUE     8
MISSING_ITEM              6
WRONG_ITEM_RECEIVED       5
Name: count, dtype: int64


In [66]:
golden_df = annotation_df[
    annotation_df["human_intent"].notna() &
    (annotation_df["human_intent"] != "")
].copy()

golden_columns = [
    "customer_tweet_id",
    "customer_text",
    "customer_text",
    "human_intent",
    "difficulty",
    "notes"
]

golden_df = golden_df[golden_columns].copy()

golden_path = Path("../data/golden/amazon_golden_set.csv")

golden_df.to_csv(
    golden_path,
    index=False
)

print(f"Saved {len(golden_df)} examples")
print(golden_path)

Saved 213 examples
..\data\golden\amazon_golden_set.csv


In [68]:
print("Size:", len(golden_df))
print("Duplicate IDs:", golden_df["customer_tweet_id"].duplicated().sum())
print("Duplicate texts:", golden_df["customer_text"].duplicated().sum())

print("\nIntent distribution:")
print(golden_df["human_intent"].value_counts())

Size: 213
Duplicate IDs: 0
Duplicate texts: 0

Intent distribution:
human_intent
OTHER                    37
ORDER_CANCELLATION       19
REFUND_STATUS            17
DELIVERY_NOT_RECEIVED    16
DELIVERY_TRACKING        16
ACCOUNT_ACCESS           15
PRODUCT_ISSUE            13
ACCOUNT_SECURITY         12
ORDER_STATUS             12
DELIVERY_DELAY           11
RETURN_REPLACEMENT        9
UNEXPECTED_CHARGE         9
CASHBACK                  8
DELIVERY_DRIVER_ISSUE     8
MISSING_ITEM              6
WRONG_ITEM_RECEIVED       5
Name: count, dtype: int64
